# 03 — MLP Translation Head Training

This notebook trains the MLP translation head that maps frozen Concerto features → CLIP text embedding space for semantic segmentation on S3DIS Area 5.

**Prerequisites:** Feature extraction (notebook 02) must be complete — the `features/s3dis_area5/` directory should contain 68 `.npz` files (~31 GB).

**Runtime:** ~30–60 min on Colab T4.

### 1. Setup & Mount Drive

In [90]:
from google.colab import drive
drive.mount('/content/drive')

import os

REPO_DIR = '/content/Deep_learning_project'
BRANCH = 'colab_test_r'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Gandata/Deep_learning_project.git {REPO_DIR}


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [91]:
%cd {REPO_DIR}
!git fetch origin
!git checkout -B {BRANCH} origin/{BRANCH}
!git pull origin {BRANCH}
!git branch --show-current
!git log -1 --oneline

/content/Deep_learning_project
Branch 'colab_test_r' set up to track remote branch 'colab_test_r' from 'origin'.
Reset branch 'colab_test_r'
Your branch is up to date with 'origin/colab_test_r'.
From https://github.com/Gandata/Deep_learning_project
 * branch            colab_test_r -> FETCH_HEAD
Already up to date.
colab_test_r
2b18713 (HEAD -> colab_test_r, origin/colab_test_r) Pre-allocated GPU Buffer


In [92]:
#!git reset --hard origin/colab_test_r

In [93]:
!git clone https://github.com/Pointcept/Concerto.git /content/Concerto 2>/dev/null || echo 'Concerto already cloned'

Concerto already cloned


### 2. Symlink Data & Checkpoints

In [94]:
# Symlink Drive data
!rm -f data checkpoints features pretrained
!ln -sf /content/drive/MyDrive/DL_Project/data ./data
!ln -sf /content/drive/MyDrive/DL_Project/checkpoints ./checkpoints
!ln -sf /content/drive/MyDrive/DL_Project/features ./features

In [95]:
# Verify extracted features exist
!echo '--- Feature files ---'
!ls features/s3dis_area5/*.npz | wc -l
!echo 'files found'
!du -sh features/s3dis_area5/

--- Feature files ---
68
files found
31G	features/s3dis_area5/


### 3. Setup Environment

In [96]:
%cd /content/Deep_learning_project/notebooks/

/content/Deep_learning_project/notebooks


In [97]:
!uv python install 3.10.12
!uv sync --python 3.10.12

Python 3.10.12 is already installed
Resolved 147 packages in 463ms
Uninstalled 2 packages in 5ms
 - einops==0.8.2
 - flash-attn==2.8.3


In [98]:
!uv add flash-attn --no-build-isolation

Resolved 149 packages in 593ms
Installed 2 packages in 7ms
 + einops==0.8.2
 + flash-attn==2.8.3


In [99]:
# Setup HF token for open_clip
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

with open(".env", "w") as f:
    f.write(f"HF_TOKEN={HF_TOKEN}\n")

### 4. (Optional) Copy features to local SSD for faster I/O
Google Drive I/O is slow. Copying the features to the Colab local disk speeds up training significantly.

Skip this cell if features are already local or you're low on disk space.

In [100]:
%%time
import shutil, os

LOCAL_FEATURES = '/content/features_local/s3dis_area5'
DRIVE_FEATURES = '/content/Deep_learning_project/features/s3dis_area5'

if not os.path.exists(LOCAL_FEATURES):
    os.makedirs(LOCAL_FEATURES, exist_ok=True)
    print(f'Copying features from Drive to local SSD...')
    !cp -v {DRIVE_FEATURES}/*.npz {LOCAL_FEATURES}/
    print('Done!')
else:
    print(f'Local features already exist at {LOCAL_FEATURES}')

!ls {LOCAL_FEATURES}/*.npz | wc -l
!echo 'feature files on local SSD'

Local features already exist at /content/features_local/s3dis_area5
68
feature files on local SSD
CPU times: user 3.13 ms, sys: 2.28 ms, total: 5.41 ms
Wall time: 109 ms


### 5. Update config to point to local features

If you copied features to local SSD in step 4, we patch the config to use the faster local path.

In [101]:
import yaml

CONFIG_PATH = '/content/Deep_learning_project/configs/train_mlp_s3dis.yaml'
LOCAL_FEATURES = '/content/features_local/s3dis_area5'

with open(CONFIG_PATH, 'r') as f:
    cfg = yaml.safe_load(f)

# Use local SSD path if available, otherwise keep Drive path
import os
if os.path.exists(LOCAL_FEATURES):
    cfg['data']['train_features_path'] = LOCAL_FEATURES
    print(f'Using LOCAL features: {LOCAL_FEATURES}')
else:
    print(f'Using DRIVE features: {cfg["data"]["train_features_path"]}')

with open(CONFIG_PATH, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False, sort_keys=False)

print('\n--- Current config ---')
!cat {CONFIG_PATH}

Using LOCAL features: /content/features_local/s3dis_area5

--- Current config ---
data:
  train_features_path: /content/features_local/s3dis_area5
  val_features_path: null
  label_embeddings_path: /content/Deep_learning_project/data/s3dis_processed/label_to_clip_embeddings.npy
  normalize_features: true
  train_room_type_fraction: 1.0
  train_room_type_fraction_seed: 42
  inspect_files: false
  verbose_inspection: false
  val_holdout_fraction: 0.15
  val_holdout_seed: 42
  label_texts:
  - ceiling of a room
  - floor of a room
  - wall of a room
  - beam on the ceiling
  - column or pillar
  - window
  - door
  - sofa or couch
  - table or desk
  - chair
  - bookcase or shelf
  - whiteboard or board
  - clutter or miscellaneous object
clip:
  model_name: ViT-B-32
  pretrained: openai
  templates:
  - a photo of a {}
  - a 3D point cloud of a {}
  - a {}
model:
  input_dim: 896
  hidden_dims:
  - 512
  - 512
  output_dim: 512
  dropout: 0.1
  activation: gelu
  normalize_output: true
t

### 6. Run Training

This runs the MLP translation head training with:
- **Hybrid loss** (MSE + cosine) — aligns training with cosine-similarity evaluation
- **30 epochs** with cosine LR annealing
- **15% holdout validation** stratified by room type
- **torch.compile** for ~5-10% speedup
- **100% of Area 5 rooms** (no fraction sampling)

Estimated time: **30–60 min on T4**.

In [102]:
%%time
# Train the MLP translation head
!PYTHONPATH=/content/Deep_learning_project:/content/Concerto \
  uv run python /content/Deep_learning_project/src/train.py \
  --config /content/Deep_learning_project/configs/train_mlp_s3dis.yaml

Using device: cuda
Holdout validation split: 15% of files per room type (seed=42)
  WC: 1 train / 1 val (of 2)
  conferenceRoom: 2 train / 1 val (of 3)
  hallway: 12 train / 3 val (of 15)
  lobby: 1 file → all train (no val)
  office: 35 train / 7 val (of 42)
  pantry: 1 file → all train (no val)
  storage: 3 train / 1 val (of 4)
Holdout result: 55 train files, 13 val files
GPU-buffered training enabled: budget=10.0 GB, AMP=on
Using CosineAnnealingLR scheduler (T_max=30, eta_min=1e-05)
Train files:   55
Train samples: 0
Val files:     13
Val samples:   0
Feature dim:   896
Model params:  984,576
CPU times: user 1.5 s, sys: 200 ms, total: 1.71 s
Wall time: 13min 4s


### 7. Plot Training Curves

In [103]:
import json
import matplotlib.pyplot as plt

METRICS_PATH = '/content/Deep_learning_project/checkpoints/mlp_s3dis/history.json'

with open(METRICS_PATH, 'r') as f:
    history = json.load(f)

epochs = [h['epoch'] for h in history]
train_loss = [h['train_loss'] for h in history]
val_loss = [h['val_loss'] for h in history if h['val_loss'] is not None]
val_epochs = [h['epoch'] for h in history if h['val_loss'] is not None]
lrs = [h.get('lr', None) for h in history]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss plot
axes[0].plot(epochs, train_loss, label='Train Loss', linewidth=2)
if val_loss:
    axes[0].plot(val_epochs, val_loss, label='Val Loss', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# LR plot
if lrs[0] is not None:
    axes[1].plot(epochs, lrs, linewidth=2, color='green')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Learning Rate')
    axes[1].set_title('Learning Rate Schedule')
    axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/Deep_learning_project/checkpoints/mlp_s3dis/training_curves.png', dpi=150)
plt.show()

print(f"\nFinal train loss: {train_loss[-1]:.6f}")
if val_loss:
    print(f"Final val loss:   {val_loss[-1]:.6f}")
    print(f"Best val loss:    {min(val_loss):.6f}")

FileNotFoundError: [Errno 2] No such file or directory: '/content/Deep_learning_project/checkpoints/mlp_s3dis/history.json'

### 8. Evaluate (OA & mIoU)

In [104]:
%%time
# Evaluate using the best checkpoint
CHECKPOINT = '/content/Deep_learning_project/checkpoints/mlp_s3dis/best_model.pth'
FEATURES_DIR = '/content/Deep_learning_project/features/s3dis_area5'

# Use local features if available
import os
if os.path.exists('/content/features_local/s3dis_area5'):
    FEATURES_DIR = '/content/features_local/s3dis_area5'

!PYTHONPATH=/content/Deep_learning_project:/content/Concerto \
  uv run python /content/Deep_learning_project/src/evaluate.py \
  --features_dir {FEATURES_DIR} \
  --checkpoint {CHECKPOINT} \
  --clip_model ViT-B-32 \
  --clip_pretrained openai

Loading CLIP model ViT-B-32 (openai)...
open_clip_model.safetensors: 100% 605M/605M [00:04<00:00, 140MB/s]
/content/Deep_learning_project/notebooks/.venv/lib/python3.10/site-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(
/content/Deep_learning_project/src/evaluate.py:50: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly a

### 9. Copy checkpoints back to Drive

Make sure the trained model is safely stored on Drive.

In [ ]:
!mkdir -p /content/drive/MyDrive/DL_Project/checkpoints/mlp_s3dis/
!cp -v /content/Deep_learning_project/checkpoints/mlp_s3dis/*.pth \
       /content/drive/MyDrive/DL_Project/checkpoints/mlp_s3dis/
!cp -v /content/Deep_learning_project/checkpoints/mlp_s3dis/history.json \
       /content/drive/MyDrive/DL_Project/checkpoints/mlp_s3dis/
!cp -v /content/Deep_learning_project/checkpoints/mlp_s3dis/training_curves.png \
       /content/drive/MyDrive/DL_Project/checkpoints/mlp_s3dis/ 2>/dev/null || true
print('Checkpoints backed up to Drive!')

Change yaml

In [ ]:
yaml_content = """
data:
  train_features_path: "/content/Deep_learning_project/features/s3dis_area5"
  val_features_path: null
  label_embeddings_path: "/content/Deep_learning_project/data/s3dis_processed/label_to_clip_embeddings.npy"

  normalize_features: true
  train_room_type_fraction: 1.0
  train_room_type_fraction_seed: 42

  # Inspection settings
  inspect_files: false
  verbose_inspection: false

  # Hold out 15% of rooms (stratified by room type) as validation
  val_holdout_fraction: 0.15
  val_holdout_seed: 42

  label_texts:
    - "ceiling of a room"
    - "floor of a room"
    - "wall of a room"
    - "beam on the ceiling"
    - "column or pillar"
    - "window"
    - "door"
    - "sofa or couch"
    - "table or desk"
    - "chair"
    - "bookcase or shelf"
    - "whiteboard or board"
    - "clutter or miscellaneous object"

clip:
  model_name: "ViT-B-32"
  pretrained: "openai"
  templates:
    - "a photo of a {}"
    - "a 3D point cloud of a {}"
    - "a {}"

model:
  input_dim: 896
  hidden_dims: [512, 512]
  output_dim: 512
  dropout: 0.1
  activation: "gelu"
  normalize_output: true

training:
  seed: 42
  device: null
  epochs: 30
  batch_size: 65536
  learning_rate: 0.001
  weight_decay: 0.0001
  loss: "mse+cosine"
  mse_weight: 1.0
  cosine_weight: 1.0
  save_every: 5
  save_best: true
  resume_from: null
  checkpoint_dir: "/content/Deep_learning_project/checkpoints/mlp_s3dis"
  metrics_path: "/content/Deep_learning_project/checkpoints/mlp_s3dis/history.json"

  # LR scheduler: cosine annealing to 1e-5 over all epochs
  scheduler: "cosine"
  scheduler_min_lr: 0.00001

  # torch.compile the model for ~5-10% speedup on PyTorch 2.x
  compile_model: true

  # GPU-resident data buffer: load multiple rooms onto GPU at once (~12 GB)
  gpu_buffer: true
  gpu_budget_gb: 8

  # AMP (Automatic Mixed Precision): T4 tensor cores
  use_amp: true
"""

import os

# Ensure the directory exists
os.makedirs("/content/Deep_learning_project/configs", exist_ok=True)

# Write the file
with open("/content/Deep_learning_project/configs/train_mlp_s3dis.yaml", "w") as f:
    f.write(yaml_content.strip())

print("YAML file created successfully at /content/Deep_learning_project/configs/train_mlp_s3dis.yaml")